In [25]:
import pandas as pd

In [26]:
df = pd.read_csv('final_data.csv')
# Dropping unwanted columns
df.drop('OrderValue', axis=1, inplace=True)
df.drop('OrderID', axis=1, inplace=True)
df.drop('CustomerID', axis=1, inplace=True)
df.drop('ProductID', axis=1, inplace=True)
df.drop('SignupDate', axis=1, inplace=True)

df.head()

,OrderDate,Quantity,Discount,PaymentMethod,Status,Age,City,CustomerSegment,ProductName,Category,UnitPrice,Sales
0,2025-08-28,4.0,10.0,Gateway,Completed,36.0,Qom,Regular,USB-C Cable,Accessories,9.0,32.0
1,2024-05-31,1.0,10.0,Wallet,Completed,49.0,Kerman,Regular,Backpack,Accessories,42.0,38.0
2,2025-08-10,1.0,20.0,CardToCard,Completed,36.0,Tehran,Regular,Mechanical Keyboard,Electronics,62.0,50.0
3,2024-10-11,1.0,10.0,Gateway,Completed,45.0,Shiraz,Regular,Office Chair,Home Office,180.0,162.0
4,2024-02-19,2.0,0.0,Gateway,Completed,24.0,Karaj,Regular,Phone Case,Accessories,14.0,28.0


In [27]:
# Fixing incorrect data types
numerical_cols = ['Quantity', 'Discount', 'Age', 'UnitPrice', 'Sales']
categorical_cols = ['PaymentMethod', 'City', 'CustomerSegment', 'ProductName', 'Category']
date_cols = ['OrderDate']

for col in numerical_cols:
    df[col] = df[col].astype('int32')

for col in categorical_cols:
    df[col] = df[col].astype('category')

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49222 entries, 0 to 49221
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderDate        49222 non-null  datetime64[us]
 1   Quantity         49222 non-null  int32         
 2   Discount         49222 non-null  int32         
 3   PaymentMethod    49222 non-null  category      
 4   Status           49222 non-null  str           
 5   Age              49222 non-null  int32         
 6   City             49222 non-null  category      
 7   CustomerSegment  49222 non-null  category      
 8   ProductName      49222 non-null  category      
 9   Category         49222 non-null  category      
 10  UnitPrice        49222 non-null  int32         
 11  Sales            49222 non-null  int32         
dtypes: category(5), datetime64[us](1), int32(5), str(1)
memory usage: 2.3 MB


In [29]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

feature = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(), categorical_cols)
    ]
)
feature

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

## Task 1 --> Support Vector Machine

In [31]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df.drop('Status', axis=1)
Y = df['Status']

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

X_train = feature.fit_transform(X_train)
X_test = feature.transform(X_test)


## SVC as kernel = "rbf"
svc_rbf = SVC(kernel='rbf')
svc_rbf.fit(X_train, y_train)

y_pred_rbf = svc_rbf.predict(X_test)
rbf_accuracy = accuracy_score(y_test, y_pred_rbf)

## SVC as kernel = "linear"
svc_linear = SVC(kernel='linear')
svc_linear.fit(X_train, y_train)

y_pred_linear = svc_linear.predict(X_test)
linear_accuracy = accuracy_score(y_test, y_pred_linear)

print("Linear Kernel Accuracy:", linear_accuracy)
print("RBF Kernel Accuracy:", rbf_accuracy)

Linear Kernel Accuracy: 0.9162011173184358
RBF Kernel Accuracy: 0.9162011173184358


## Task 2 --> Decision Tree Algorithm

In [32]:
from sklearn.tree import DecisionTreeClassifier

tree_low = DecisionTreeClassifier(max_depth=2, random_state=42)

tree_low.fit(X_train, y_train)

y_train_pred_low = tree_low.predict(X_train)
y_test_pred_low = tree_low.predict(X_test)

train_accu_low = accuracy_score(y_train, y_train_pred_low)
test_accu_low = accuracy_score(y_test, y_test_pred_low)

print("Train Accuracy:", train_accu_low)
print("Test Accuracy:", test_accu_low)

Train Accuracy: 0.9207405338141554
Test Accuracy: 0.9162011173184358


In [33]:
tree_high = DecisionTreeClassifier(max_depth=20, random_state=42)

tree_high.fit(X_train, y_train)

y_train_pred_high = tree_high.predict(X_train)
y_test_pred_high = tree_high.predict(X_test)

train_accu_high = accuracy_score(y_train, y_train_pred_high)
test_accu_high = accuracy_score(y_test, y_test_pred_high)

print("Train Accuracy:", train_accu_high)
print("Test Accuracy:", test_accu_high)

Train Accuracy: 0.9616019503771237
Test Accuracy: 0.8620619603859827


## Visualizing the tree structure

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 10))
plot_tree(tree_low, filled=True, class_names=[str(c) for c in tree_low.classes_])
plt.show()

## Task 3 --> Train , Validation and Test Split

In [36]:
## Splitting now in training data for Validation
X_train_new, X_val, y_train_new, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print("Training Set:", X_train_new.shape)
print("Validation Set:", X_val.shape)
print("Test Set:", X_test.shape)

Training Set: (31501, 48)
Validation Set: (7876, 48)
Test Set: (9845, 48)


In [37]:
depths = [2, 3, 5, 10, 20]

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train_new, y_train_new)

    y_val_pred = tree.predict(X_val)

    val_accuracy = accuracy_score(y_val, y_val_pred)
    print(f'max depth : {depth}, Validation accuracy: {val_accuracy}')

max depth : 2, Validation accuracy: 0.920264093448451
max depth : 3, Validation accuracy: 0.920264093448451
max depth : 5, Validation accuracy: 0.920264093448451
max depth : 10, Validation accuracy: 0.9106145251396648
max depth : 20, Validation accuracy: 0.8597003555104114


## Now Evaluating Best final Tree with Test Set

In [38]:
final_tree = DecisionTreeClassifier(max_depth=2, random_state=42)

final_tree.fit(X_train_new, y_train_new)

y_pred = final_tree.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

test_accuracy

0.9162011173184358

## Task 4 --> Cross-Validation

In [45]:
from sklearn.model_selection import KFold, cross_val_score

tree_cv = DecisionTreeClassifier(max_depth=2, random_state=42)
kFold = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(tree_cv, X_train, y_train, cv=kFold, scoring='accuracy')
print("Cross-Validation Scores:", cv_scores)

tree_cv.fit(X_train, y_train)

y_pred_cv = tree_cv.predict(X_test)

accuracy_cv = accuracy_score(y_test, y_pred_cv)

print("Single Train-Test Accuracy:", accuracy_cv)
print("Average Cross-Validation Accuracy:", cv_scores.mean())

Cross-Validation Scores: [0.92026409 0.92229558 0.91657143 0.92342857 0.92114286]
Single Train-Test Accuracy: 0.9162011173184358
Average Cross-Validation Accuracy: 0.9207405064209533


## Task 5 --> Bagging and Boosting
#### Bagging trains multiple models independently on different random samples of the training data and combines their predictions.
#### Boosting trains models sequentially. Each new model focuses more on the observations that previous models predicted incorrectly.

In [46]:
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier

bag = BaggingClassifier(n_estimators=50)

bag.fit(X_train, y_train)

y_pred_bag = bag.predict(X_test)

accuracy_score(y_test, y_pred_bag)

0.905332656170645

In [49]:
ada = AdaBoostClassifier(n_estimators=50)

ada.fit(X_train, y_train)

y_pred_ada = ada.predict(X_test)

accuracy_score(y_test, y_pred_ada)

0.9162011173184358

##### AdaBoost performed better than Bagging on the test dataset. 
##### AdaBoost achieved an accuracy of 91.6%, while Bagging achieved 90.5%.
##### Therefore, AdaBoost performed approximately 1.1 percentage points better than Bagging.

## Task 6 --> Random Forest 

In [52]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
# Random Forest Accuracy
rf_accuracy = accuracy_score(y_test, y_pred_rf)

# Single Decision Tree
tree = DecisionTreeClassifier(max_depth=2)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)
# Decision Tree Accuracy
tree_accuracy = accuracy_score(y_test, y_pred_tree)

bag = BaggingClassifier(n_estimators=50)
bag.fit(X_train, y_train)
y_pred_bag = bag.predict(X_test)
bag_accuracy = accuracy_score(y_test, y_pred_bag)


print("Decision Tree Accuracy :", tree_accuracy)
print("Bagging Accuracy :", bag_accuracy)
print("Random Forest Accuracy :", rf_accuracy)


Decision Tree Accuracy : 0.9162011173184358
Bagging Accuracy : 0.9060436769933977
Random Forest Accuracy : 0.8995429151853733


In [53]:
# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': feature.get_feature_names_out(),
    'Importance': rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print(feature_importance)

                                 Feature  Importance
2                               num__Age    0.383994
4                             num__Sales    0.130955
1                          num__Discount    0.092181
0                          num__Quantity    0.052252
7             cat__PaymentMethod_Gateway    0.023547
20          cat__CustomerSegment_Regular    0.022012
19              cat__CustomerSegment_New    0.021218
3                         num__UnitPrice    0.021050
5          cat__PaymentMethod_CardToCard    0.019645
18                      cat__City_Tehran    0.019614
8              cat__PaymentMethod_Wallet    0.019419
13                     cat__City_Mashhad    0.015342
6                cat__PaymentMethod_Cash    0.014164
11                       cat__City_Karaj    0.013877
17                      cat__City_Tabriz    0.013858
21              cat__CustomerSegment_VIP    0.013457
10                     cat__City_Isfahan    0.012632
16                      cat__City_Shiraz    0.